In [1]:
import os
import pyodbc
import pandas as pd

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)

In [4]:
df_Data_Dim_Dan_toc = pd.read_csv("./Data_Dim_Dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
print(df_Data_Dim_Dan_toc)

    Mã               Tên                                       Tên gọi khác
0    1              Kinh                                               Việt
1    2               Tày          Thổ, Ngạn, Phén, Thù Lao, Pa Dí, Tày Khao
2    3              Thái  Tày Đăm, Tày Mười, Tày Thanh, Mán Thanh, Hàng ...
3    4               Hoa  Hán, Triều Châu, Phúc Kiến, Quảng Đông, Hải Na...
4    5            Khơ-me             Cur, Cul, Cu, Thổ, Việt gốc Miên, Krôm
5    6             Mường               Mol, Mual, Mọi, Mọi Bi, Ao Tá, Ậu Tá
6    7              Nùng  Xuồng, Giang, Nùng An, Phàn Sinh, Nùng Cháo, N...
7    8             HMông  Mèo, Hoa, Mèo Xanh, Mèo Đỏ, Mèo Đen, Ná Mẻo, M...
8    9               Dao  Mán, Động, Trại, Xá, Dìu, Miên, Kiềm, Miền, Qu...
9   10           Gia-rai    Giơ-rai, Tơ-buăn, Chơ-rai, Hơ-bau, Hđrung, Chor
10  11              Ngái                            Xín, Lê, Đản, Khách Gia
11  12              Ê-đê  Ra-đê, Đê, Kpạ, A-đham, Krung, Ktul, Đliê Ruê,...
12  13      

In [5]:
query_Dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)
print(df_Dan_toc)

    Id  Dan_toc
0    1     Kinh
1    2    Mường
2    3      Tày
3    4     Thái
4    5      Hoa
..  ..      ...
68  71     Ê Đê
69  72      Thổ
70  73    Kờ Ho
71  74     Jrai
72  75  Châu mạ

[73 rows x 2 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_22692\749085140.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)


In [6]:
df_Dan_toc['Mapping Mã'] = None
df_Dan_toc['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_Dan_toc['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_Dan_toc['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_Dan_toc['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_Dan_toc.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_Dan_toc.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_Dan_toc.at[i, 'Mapping Mã'] = 56
print(df_Dan_toc)

    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

In [9]:
file_path = './Mapping_Dan_toc.csv'
if os.path.exists(file_path):
    os.remove(file_path)
df_Dan_toc.to_csv(file_path, index=False, encoding='utf-8-sig')